# Part 2 — Destination & Mode Choice

Loads the population and activity state saved by **Part 1**,
then assigns destinations, validates day plans, builds tours,
and runs LLM-powered mode choice.

**Requires** an `ANTHROPIC_API_KEY` environment variable and
`../data/processed/pipeline_state.json` produced by Part 1.

## 1. Load state from Part 1

In [ ]:
import json
from collections import Counter
from pathlib import Path

from aibm import Agent, Household, Zone
from aibm.activity import Activity
from aibm.agent import ModeOption
from aibm.day_plan import DayPlan
from aibm.llm import AnthropicClient

input_path = Path("../data/processed/pipeline_state.json")
state = json.loads(input_path.read_text())

# --- Zones -------------------------------------------------
zones = [
    Zone(
        id=z["id"],
        name=z["name"],
        x=z["x"],
        y=z["y"],
        land_use=z["land_use"],
    )
    for z in state["zones"]
]
zone_lookup = {z.id: z for z in zones}

# --- Travel times ------------------------------------------
travel_times: dict[tuple[str, str], dict[str, int]] = {
    tuple(k.split(",")): v  # type: ignore[misc]
    for k, v in state["travel_times"].items()
}

# --- Households & agents -----------------------------------
households: list[Household] = []
all_agents: list[Agent] = []

for hh_data in state["households"]:
    members = [
        Agent(
            id=m["id"],
            name=m["name"],
            model=m["model"],
            age=m["age"],
            employment=m["employment"],
            has_license=m["has_license"],
            home_zone=m["home_zone"],
            work_zone=m["work_zone"],
            school_zone=m["school_zone"],
            persona=m["persona"],
        )
        for m in hh_data["members"]
    ]
    hh = Household(
        id=hh_data["id"],
        members=members,
        home_zone=hh_data["home_zone"],
        num_vehicles=hh_data["num_vehicles"],
        income_level=hh_data["income_level"],
    )
    households.append(hh)
    all_agents.extend(members)

hh_lookup = {
    a.id: hh
    for hh in households
    for a in hh.members
}

# --- Activities & day plans --------------------------------
agent_activities: dict[str, list[Activity]] = {}
agent_plans: dict[str, DayPlan] = {}

for agent in all_agents:
    acts = [
        Activity(
            type=a["type"],
            location=a["location"],
            start_time=a["start_time"],
            end_time=a["end_time"],
            is_flexible=a["is_flexible"],
        )
        for a in state["agent_activities"][agent.id]
    ]
    agent_activities[agent.id] = acts

    plan_acts = [
        Activity(
            type=a["type"],
            location=a["location"],
            start_time=a["start_time"],
            end_time=a["end_time"],
            is_flexible=a["is_flexible"],
        )
        for a in state["agent_plans"][agent.id]
    ]
    agent_plans[agent.id] = DayPlan(activities=plan_acts)

client = AnthropicClient()

print(
    f"Loaded: {len(households)} households, "
    f"{len(all_agents)} agents"
)

## 2. Choose destinations

In [ ]:
for agent in all_agents:
    for activity in agent_activities[agent.id]:
        if activity.location is None:
            agent.choose_destination(
                activity,
                candidates=zones,
                client=client,
            )
            print(
                f"{agent.name} -> {activity.type} "
                f"at {activity.location}"
            )

## 3. Validate day plans

In [ ]:
for agent in all_agents:
    day_plan = agent_plans[agent.id]
    warnings = day_plan.validate()
    if warnings:
        print(f"{agent.name}: {len(warnings)} warning(s)")
        for w in warnings:
            print(f"  ! {w}")
    else:
        print(f"{agent.name}: OK")

## 4. Build tours

In [ ]:
for agent in all_agents:
    day_plan = agent_plans[agent.id]
    agent.build_tours(day_plan)
    n_tours = len(day_plan.tours)
    n_trips = len(day_plan.trips)
    print(
        f"\n{agent.name}: "
        f"{n_tours} tour(s), {n_trips} trip(s)"
    )
    for i, tour in enumerate(day_plan.tours):
        print(
            f"  Tour {i + 1} "
            f"(closed={tour.is_closed}):"
        )
        for trip in tour.trips:
            print(
                f"    {trip.origin} -> "
                f"{trip.destination}"
            )

## 5. Choose mode per tour

In [ ]:
all_mode_choices = []

for agent in all_agents:
    hh = hh_lookup[agent.id]
    day_plan = agent_plans[agent.id]

    for tour in day_plan.tours:
        first_trip = tour.trips[0]
        pair = (first_trip.origin, first_trip.destination)
        tt = travel_times.get(pair, {"walk": 30})

        # Filter car if no licence or 0 vehicles
        options = []
        for mode, minutes in tt.items():
            if mode == "car" and (
                not agent.has_license
                or hh.num_vehicles == 0
            ):
                continue
            options.append(
                ModeOption(
                    mode=mode,
                    travel_time=minutes,
                )
            )

        if not options:
            options = [
                ModeOption(mode="walk", travel_time=30)
            ]

        choice = agent.choose_tour_mode(
            tour, options, client=client, household=hh
        )
        all_mode_choices.append(choice)
        stops = " -> ".join(
            t.destination for t in tour.trips
        )
        print(
            f"{agent.name}: {stops} "
            f"by {choice.option.mode}"
        )

## 6. Summary — mode share

In [ ]:
all_trips = [
    trip
    for agent in all_agents
    for trip in agent_plans[agent.id].trips
]

mode_counts = Counter(t.mode for t in all_trips)
total = len(all_trips)

print(f"Total trips: {total}\n")
print("Mode share:")
for mode, count in mode_counts.most_common():
    print(
        f"  {mode:8s}: {count:3d}  "
        f"({count / total:.0%})"
    )

## 7. Save final results

In [ ]:
output_dir = Path("../data/processed")

results = {
    "trips": [
        {
            "origin": t.origin,
            "destination": t.destination,
            "mode": t.mode,
            "departure_time": t.departure_time,
        }
        for t in all_trips
    ],
    "mode_share": {
        mode: count
        for mode, count in mode_counts.most_common()
    },
    "total_trips": total,
}

output_path = output_dir / "pipeline_results.json"
output_path.write_text(json.dumps(results, indent=2))
print(f"Saved final results to {output_path}")